# Pipeline OCR OV — v29 (base v28, elle-même base v27)

**v28 :** la case 70 de l'OV est recopiée mot pour mot (`reference_paiement`) ; mois / année / tranche / complément
en sont déduits (dans TL-One par `tlone.periode.lire`, qui fait foi).

**v29 — corrections des réserves listées plus bas (aucun changement du modèle ni de l'inférence) :**
1. `norm_montant` robuste : `23.340.43` → 23340.43 ; `312:255:00` → 312255.00 ; `410 542,13` → 410542.13 ;
   `1.000` → 1000 ; le dernier séparateur n'est décimal que s'il est suivi de 1 ou 2 chiffres.
2. `nature_paiement_case` (OV) et `nombre_jours_absence` (Annexe II), demandés par le prompt, ne sont plus écartés
   par les listes blanches ; ils sortent dans l'Excel (« Nature paiement (case) », « Nombre jours absence »).
3. Annexe I : `date_naissance` et `résidence` sortent dans l'Excel (« Date de naissance », « Résidence »).
4. Numéro de page de chaque pièce mémorisé (`page` dans chaque pièce du JSON, colonne META « Pages par pièce »
   `OV:1, ANNEXE_I:2, …`) : TL-One ouvre directement la bonne page du PDF ; les doublons indiquent leurs pages.
5. Tokens de sortie comptés sans le padding du batch ; version du pipeline écrite dans chaque JSON.
Les JSON v29 vont dans `transferts_out/json_dossiers_v29` (les PDF sont relus une fois).
À valider en non-régression avant de remplacer v27 : `TLONE_OV_NOTEBOOK=pipeline_ocr_vl29.ipynb`.

---

# Pipeline OCR V12 — restitution des captures du 18 octobre 2026

**Version reconstituée avec réserves — pas une copie certifiée à l'identique.**
Base : `pipeline ocr v12(1).ipynb`, complétée par les 27 images reçues, IMG_8475 à IMG_8502 sauf IMG_8483. Le nom de l'onglet photographié est « pipeline ocr VLM27 (3) (1).ipynb » ; le nom V12 demandé est conservé pour le livrable.

## Provenance et limites
- Sections 2 à 11 : rapprochées des captures ; les changements fonctionnels visibles ont été rétablis.
- Titre introductif V11, dépendances et sections 12–13 : conservés du notebook fourni, faute de captures correspondantes. Leur présence dans la dernière version n'est pas confirmée.
- IMG_8499/8500 : bloc d’inférence confirmé. Correction de `text=[text_in]` dans `ask_single` et rétablissement du nom local `messages` dans `ask_batch`.
- IMG_8483 n'a pas été reçue. La jonction `norm_periode` est couverte par IMG_8482/8484, mais des lignes supplémentaires éventuelles ne peuvent être exclues.
- IMG_8501 : `complement = m.group(1)` confirmé, réserve levée. IMG_8502 confirme les whitelists et les alias de compte.
- IMG_8489 : fin de `nature_paiement_autre_libelle` coupée après « ex » ; l'exemple non visible est omis et la parenthèse refermée. Fin de `numero_domiciliation` coupée après « Retourner les 5 valeurs séparées pa… » : le séparateur est inconnu, cette instruction tronquée est omise. Ces deux fins de ligne sont à fournir pour une restitution exacte du prompt.
- Espacement, commentaires et quelques libellés ont été retranscrits ; les sorties historiques et compteurs d'exécution ont été effacés pour ne pas simuler une exécution de cette restauration.

## Changements rétablis
- Qwen2.5-VL-7B, 6 000 tokens, taille image 2 024, zoom PDF 3.0, seuil blanc 0.95, batch 8, padding à gauche.
- Répertoire d'entrée `/mnt/data/transferts_in/06_07_08_09_2026`.
- Inférence Qwen avec `processor.apply_chat_template` ; retrait du bloc InternVL présent dans la copie ancienne.
- Montants avec séparateurs européens/anglo-saxons ; période conservée brute et extraction mois/année/tranche/complément.
- Champs et prompt actualisés pour OV, ANNEXE_I, ANNEXE_II et BULLETIN ; export de 46 colonnes dans l'ordre photographié.
- Boucle dossier par dossier avec checkpoint JSON immédiat et reprise sur les JSON existants.

## Vérification et comportements conservés
Syntaxe Python et structure du notebook vérifiées, avec tests locaux des fonctions de normalisation et de l'export. **Aucune inférence réelle exécutée** : modèle, PDF métier et environnement GPU Domino indisponibles ici.

Les comportements suivants sont visibles dans les captures et n'ont pas été corrigés discrètement :
- `nature_paiement_case` et `nombre_jours_absence` demandés dans le prompt sont écartés par les whitelists.
- `date_naissance` et `résidence` restent dans le JSON ANNEXE_I mais ne figurent pas dans l'Excel.
- `23.340.43` n'est pas normalisé par la fonction de montant photographiée (retour `None`).
- Les JSON existants sont ignorés sur le seul nom du PDF, même s'ils proviennent d'une ancienne version ; l'Excel rassemble tous les JSON du dossier de sortie.
- Le compteur de tokens de sortie du batch utilise la longueur des séquences, pouvant inclure du padding.

Pour compléter la restitution : fournir les deux fins de ligne d’IMG_8489 avec défilement horizontal et IMG_8483 si elle existe. Les réserves sur IMG_8479 et l’affectation du complément sont levées.

---


# OCR BNP — Pipeline V11 — Batch GPU + JSON checkpoint + Stats
> `Kernel → Restart Kernel and Run All Cells`

**Améliorations V11 :**
- Batch GPU : plusieurs pages traitées en parallèle (x3-4 plus rapide)
- JSON sauvegardé après chaque dossier → reprise automatique si crash
- Tokens consommés + temps enregistrés dans chaque JSON
- Numéro séquentiel dans le log


## 1. Dépendances

In [ ]:
%pip install -q pymupdf pillow transformers openpyxl
print('✅ OK')

## 2. Imports

In [ ]:
import time, json, re
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print('✅ Imports OK')

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)


## 3. Config

In [ ]:
MODEL_PATH      = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS  = 6000
IMAGE_MAX_SIZE  = 2024
PDF_ZOOM        = 3.0
BLANK_THRESHOLD = 0.95
GPU_BATCH_SIZE  = 8      # Nombre de pages traitées en parallèle — ajuster selon VRAM

INPUT_DIR  = Path("/mnt/data/transferts_in/06_07_08_09_2026")
OUTPUT_DIR = Path("/mnt/data/transferts_out")
JSON_DIR   = OUTPUT_DIR / "json_dossiers_v29"   # v29 : JSON séparés de v27/v28 (sinon les PDF déjà lus seraient sautés)
PIPELINE_VERSION = "OCR_OV_V29"
LOG_PATH   = OUTPUT_DIR / "pipeline.log"
EXCEL_PATH = OUTPUT_DIR / f"audit_transferts_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob("*.pdf"))
print(f'Device         : {DEVICE}')
print(f'GPU batch size : {GPU_BATCH_SIZE}')
print(f'JSON dir       : {JSON_DIR}')
print(f'Excel          : {EXCEL_PATH}')

## 4. Chargement modèle

In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16,
    trust_remote_code=True, low_cpu_mem_usage=True,
)
processor.tokenizer.padding_side = "left"
model.eval().to(DEVICE)
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')

In [ ]:
import transformers
print(transformers.__version__)


## 5. Utilitaires PDF & inférence

In [ ]:
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)


def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert("L"))
    return (arr > 240).sum() / arr.size >= threshold


def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc    = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages  = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = resize(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
        pages.append({"index": i, "image": img})
    doc.close()
    return pages


def parse_json(text: str) -> dict:
    try:
        m = re.search(r'\{.*\}', text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}


def ask_single(prompt: str, image: Image.Image) -> dict:
    """Traite 1 image seule. Retourne {text, tokens_in, tokens_out, elapsed}."""
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': prompt},
    ]}]
    text_in = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_in], images=[image], return_tensors='pt'
    ).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0
    generated = out[0][inputs['input_ids'].shape[1]:]
    text = processor.decode(generated, skip_special_tokens=True,
                            clean_up_tokenization_spaces=True)
    return {
        'text': text,
        'tokens_in': int(inputs['input_ids'].shape[1]),
        'tokens_out': int(len(generated)),
        'elapsed': round(elapsed, 2),
    }


def ask_batch(prompt: str, images: list) -> list:
    """Traite N images en un seul appel GPU.
    Retourne une liste de {text, tokens_in, tokens_out, elapsed}.
    Même prompt pour toutes les images du batch.
    """
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0])]
    texts_in = []
    for img in images:
        messages = [{'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text', 'text': prompt},
        ]}]
        texts_in.append(
            processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )
    inputs = processor(
        text=texts_in,
        images=images,
        return_tensors='pt',
        padding=True,
    ).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    elapsed_total = time.time() - t0
    in_len = inputs['input_ids'].shape[1]
    results = []
    for i in range(len(images)):
        generated = out[i][in_len:]
        text = processor.decode(generated, skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)
        # Tokens in réels (sans padding)
        attn = inputs.get('attention_mask')
        tok_in = int(attn[i].sum().item()) if attn is not None else in_len
        # Tokens out réels : sans le padding / EOS de fin ajoutés pour aligner le batch
        pad_ids = {processor.tokenizer.eos_token_id, processor.tokenizer.pad_token_id} - {None}
        tok_out = int(sum(1 for t in generated.tolist() if t not in pad_ids))
        results.append({
            'text': text,
            'tokens_in': tok_in,
            'tokens_out': tok_out,
            'elapsed': round(elapsed_total / len(images), 2),
        })
    return results

print('✅ Utilitaires OK')


## 6. Normalisation des données

In [ ]:
def norm_str(v) -> str:
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null', 'none', 'n/a') else None

def norm_upper(v) -> str:
    s = norm_str(v)
    return s.upper() if s else None

def norm_compte(v) -> str:
    s = norm_str(v)
    if not s: return None
    return re.sub(r'[^A-Za-z0-9]', '', s).upper()

def norm_montant(v) -> float:
    """v29 : séparateurs de milliers . , espace : ' ; le dernier séparateur est décimal seulement
    s'il est suivi de 1 ou 2 chiffres.  23.340.43 -> 23340.43 | 312:255:00 -> 312255.0 | 1.000 -> 1000.0"""
    if v is None:
        return None
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip().replace('\u00a0', ' ').replace('\u202f', ' ')
    s = re.sub(r'(?<=\d)\s+(?=\d)', '', s)          # espaces de milliers
    s = re.sub(r"[^\d.,:'’]", '', s)                 # chiffres + séparateurs
    s = s.strip(".,:'’")
    if not s or not re.search(r'\d', s):
        return None
    m = re.search(r"[.,:'’](\d+)$", s)
    if m and len(m.group(1)) in (1, 2):
        ent = re.sub(r"[.,:'’]", '', s[:m.start()])
        s = f"{ent or '0'}.{m.group(1)}"
    else:
        s = re.sub(r"[.,:'’]", '', s)
    try:
        return float(s)
    except Exception:
        return None


def norm_date(v) -> str:
    if not v: return None
    s = norm_str(v)
    if not s: return None
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return s
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', s)
    if m: return f'{m.group(3)}/{m.group(2)}/{m.group(1)}'
    return s

def norm_periode(v) -> str:
    # Conserver exactement la période lue par Qwen
    if v is None:
        return None
    s = str(v).strip()
    if not s or s.lower() in ('null', 'none', 'n/a'):
        return None
    return s


def extract_periode_parts(periode):
    mois = None
    annee = None
    tranche = None
    complement = None
    if not periode:
        return mois, annee, tranche, complement
    s = str(periode).upper()
    m = re.search(
        r'\b(JANVIER|JANV|JAN|FEVRIER|FÉVRIER|FEV|FÉV|MARS|'
        r'AVRIL|AVR|MAI|JUIN|JUILLET|JUIL|AOUT|AOÛT|'
        r'SEPTEMBRE|SEPT|SEP|OCTOBRE|OCT|NOVEMBRE|NOV|'
        r'DECEMBRE|DÉCEMBRE|DEC|DÉC)\b', s
    )
    if m:
        mois = m.group(1)
    s_sans_tranche = re.sub(r'\b(P|PART|PARTIE)\s*[12]\b', ' ', s)
    m = re.search(r'(?<!\d)(20\d{2})(?!\d)', s_sans_tranche) or re.search(r'(?<!\d)(\d{2})(?!\d)', s_sans_tranche)
    if m:
        annee = m.group(1)
    if re.search(r'\b(P\s*1|PART\s*1|PARTIE\s*1)\b', s):
        tranche = 'P1'
    elif re.search(r'\b(P\s*2|PART\s*2|PARTIE\s*2)\b', s):
        tranche = 'P2'
    m = re.search(r'\b(COMPLEMENT|COMPLÉMENT|COMPL|COMP|COM|RELIQUAT)\b', s)
    if m:
        complement = m.group(1)
    return mois, annee, tranche, complement


# Whitelists — champs autorisés par type (anti-parasite)
CHAMPS_OV = {
    'type', 'monnaie', 'montant_chiffres', 'montant_lettres',
    'reference_paiement', 'periode', 'mois', 'annee', 'tranche', 'complement_ov', 'nature_paiement_case', 'nature_paiement_autre_libelle', 'date_demande', 'compte_donneur_ordre',
    'beneficiaire_nom', 'beneficiaire_compte', 'beneficiaire_adresse',
    'code_swift_banque_beneficiaire', 'nom_banque_beneficiaire'
}
CHAMPS_ANN1 = {
    'type', 'nom_prenom_employe', 'date_naissance', 'résidence', 'compte_bancaire_local',
    'nom_prenom_signataire', 'date_signature'
}
CHAMPS_ANN2 = {
    'type', 'mois_transfert', 'nom_prenom_travailleur', 'compte_bancaire_local',
    'salaire_mensuel', 'nombre_jours', 'nombre_jours_absence', 'part_transferable',
    'pays_destination', 'nom_banque_etranger', 'numero_compte_devise_etranger',
    'numero_domiciliation'
}
CHAMPS_BP = {
    'type', 'nom_prenom_salarie', 'matricule', 'periode_bulletin',
    'salaire_base', 'salaire_brut', 'retenue_ss', 'retenue_irg',
    'retenue_mutuelle', 'net_a_payer'
}


def normalise_doc(doc_type: str, data: dict) -> dict:
    d = dict(data)

    if doc_type == 'OV':
        # Aplatissement zones imbriquées
        for key in list(d.keys()):
            if isinstance(d.get(key), dict):
                d.update(d.pop(key))
        # Fallback nom alternatif compte_donneur_ordre
        if not d.get('compte_donneur_ordre'):
            for alias in ['compte_donneur_ordre', 'compte_donneur', 'compte_debiteur', 'compte_bancaire_local', 'Siège Racine Ordinal clé', 'N°DOM',
                          'numero_compte', 'compte_debiteur_ordre']:
                if d.get(alias):
                    d['compte_donneur_ordre'] = d[alias]
                    break
        # Corriger tranche = numéro de zone
        if d.get('tranche') in ('70','32','50','59','57', 70, 32, 50, 59, 57):
            d['tranche'] = None
        # Corriger periode = date (format dd/mm/yyyy)
        if d.get('periode') and re.match(r'^\d{2}/\d{2}/\d{4}$', str(d.get('periode',''))):
            d['periode'] = None
        # Whitelist
        d = {k: v for k, v in d.items() if k in CHAMPS_OV}
        # Normalisation
        d['monnaie']              = norm_upper(d.get('monnaie'))
        d['montant_chiffres']     = norm_montant(d.get('montant_chiffres'))
        d['montant_lettres']      = norm_str(d.get('montant_lettres'))
        # v28 : la case 70 est recopiée telle quelle ; periode = ce texte brut (période/tranche/complément déduits ensuite)
        d['reference_paiement']   = norm_periode(d.get('reference_paiement') or d.get('periode'))
        d['periode']              = d['reference_paiement']
        mois_auto, annee_auto, tranche_auto, complement_auto = extract_periode_parts(d.get('periode'))
        d['mois'] = norm_str(d.get('mois')) or mois_auto
        d['annee'] = norm_str(d.get('annee')) or annee_auto
        d['tranche'] = norm_str(d.get('tranche')) or tranche_auto
        d['complement_ov'] = (norm_str(d.get('complement_ov') or d.get('complement')) or complement_auto)
        d['nature_paiement_case'] = norm_str(d.get('nature_paiement_case'))
        d['nature_paiement_autre_libelle'] = norm_str(d.get('nature_paiement_autre_libelle'))
        d['date_demande']         = norm_date(d.get('date_demande'))
        d['compte_donneur_ordre']      = norm_compte(d.get('compte_donneur_ordre'))
        d['beneficiaire_nom']     = norm_upper(d.get('beneficiaire_nom'))
        d['beneficiaire_compte']  = norm_compte(d.get('beneficiaire_compte'))
        d['beneficiaire_adresse'] = norm_str(d.get('beneficiaire_adresse'))
        d['code_swift_banque_beneficiaire']                = norm_compte(d.get('code_swift_banque_beneficiaire'))
        d['nom_banque_beneficiaire']           = norm_upper(d.get('nom_banque_beneficiaire'))

    elif doc_type == 'ANNEXE_I':
        d = {k: v for k, v in d.items() if k in CHAMPS_ANN1}
        d['nom_prenom_employe'] = norm_upper(d.get('nom_prenom_employe'))
        d['date_naissance'] = norm_date(d.get('date_naissance'))
        d['résidence'] = norm_str(d.get('résidence'))
        d['compte_bancaire_local']      = norm_compte(d.get('compte_bancaire_local'))
        d['nom_prenom_signataire']= norm_upper(d.get('nom_prenom_signataire'))
        d['date_signature']       = norm_date(d.get('date_signature'))

    elif doc_type == 'ANNEXE_II':
        d = {k: v for k, v in d.items() if k in CHAMPS_ANN2}
        d['mois_transfert']         = norm_periode(d.get('mois_transfert'))
        d['nom_prenom_travailleur']             = norm_upper(d.get('nom_prenom_travailleur'))
        d['compte_bancaire_local']        = norm_compte(d.get('compte_bancaire_local'))
        d['salaire_mensuel']        = norm_montant(d.get('salaire_mensuel'))
        d['nombre_jours']           = norm_str(d.get('nombre_jours'))
        d['nombre_jours_absence']   = norm_str(d.get('nombre_jours_absence'))
        d['part_transferable']      = norm_montant(d.get('part_transferable'))
        d['pays_destination']       = norm_upper(d.get('pays_destination'))
        d['nom_banque_etranger']             = norm_str(d.get('nom_banque_etranger'))
        d['numero_compte_devise_etranger'] = norm_compte(d.get('numero_compte_devise_etranger'))
        d['numero_domiciliation']   = norm_str(d.get('numero_domiciliation'))

    elif doc_type == 'BULLETIN':
        d = {k: v for k, v in d.items() if k in CHAMPS_BP}
        d['nom_prenom_salarie'] = norm_upper(d.get('nom_prenom_salarie'))
        d['matricule']          = norm_str(d.get('matricule'))
        d['periode_bulletin']      = norm_periode(d.get('periode_bulletin'))
        d['salaire_base']       = norm_montant(d.get('salaire_base'))
        d['salaire_brut']       = norm_montant(d.get('salaire_brut'))
        d['retenue_ss']         = norm_montant(d.get('retenue_ss'))
        d['retenue_irg']        = norm_montant(d.get('retenue_irg'))
        d['retenue_mutuelle']   = norm_montant(d.get('retenue_mutuelle'))
        d['net_a_payer']        = norm_montant(d.get('net_a_payer'))

    return d


print('✅ Normalisation OK')

## 7. Prompt universel

In [ ]:
PROMPT_UNIVERSEL = 'Lis ce document bancaire.\n\nÉTAPE 1 — Identifie le type en lisant le titre principal :\n- OV : titre "ORDRE DE VIREMENT A L\'ETRANGER"\n- ANNEXE_II : titre "Annexe II" et contient "Fiche de paie spéciale relative à un transfert du salaire"\n- BULLETIN : titre "BULLETIN DE PAIE"\n- ANNEXE_I : le titre est exactement "Annexe I" (jamais "Annexe II").\nC\'est une LETTRE adressée au Directeur de l\'agence BNP.\nElle commence par "Je soussigné..." et se termine par deux signatures\n(L\'Employeur et le Requérant).\nElle NE contient PAS de tableau de salaire, pas de montants de retenues.\n\n- AUTRE : tout autre document (page vide, tableau, cachet...)\n\nÉTAPE 2 — Selon le type identifié, extrais uniquement les champs listés ci-dessous.\nSi le type est AUTRE, retourne uniquement {"type": "AUTRE"}.\n\n--- Si OV ---\nZone 32 : monnaie, montant_chiffres, montant_lettres\nZone 70 : la case située sous le libellé imprimé "Référence du paiement et observation du donneur d\'ordre".\n          reference_paiement : recopier EXACTEMENT et EN ENTIER tout le texte écrit dans cette case (imprimé, tapé ou manuscrit),\n          caractère par caractère, ligne après ligne (lignes séparées par un espace).\n          - ne pas corriger, ne pas compléter, ne pas traduire, ne pas réordonner, ne pas interpréter ;\n          - conserver majuscules/minuscules, points, espaces, /, -, chiffres, P1/P2, PART/PARTIE, COMPLEMENT, RELIQUAT… tels qu\'écrits ;\n          - ne pas recopier le libellé imprimé de la case ni le numéro 70 ;\n          - exemples de contenus possibles : JUIN.2026 PART2 / Juin 26 / Sept 25 P2 / Complément Janvier 2026 ;\n          - si la case est vide : null.\n\nZone 50 :\n    date_demande : lire la date dans les case et retourner jj/MM/AAAA\n    les traits verticaux entre les chiffres sont uniquement des séparateur de case et doivent être ignorés.\n\n    compte_donneur_ordre : lire tous les chiffres inscrit dans les case "Siège Racine Ordinal clé", dans l\'ordre de gauche à droite.\n    ignorer les traits verticaux des cases\n    retourne un seule chaine de 20 chiffres sans espace et sans "|"\n    Conserver tous les zeros initiaux\n\nZone nature_paiement : nature_paiement_case (une seule valeur parmi : virement_commercial, virement_trésorerie, urgent, non urgent, autre), nature_paiement_autre_libelle (texte EXACT à côté de la case Autre)\nZone 59 : beneficiaire_nom, beneficiaire_compte (IBAN sans espaces), beneficiaire_adresse.\nZone 57 : code_swift_banque_beneficiaire (code bic), nom_banque_beneficiaire.\n\n--- Si ANNEXE_II ---\nmois_transfert (valeur brute après "Mois de"),\nnom_prenom_travailleur,\ncompte_bancaire_local (numéro 20 chiffres),\nsalaire_mensuel,\nnombre_jours (apres le texte EXACT "Nombre de jour", null si absent),\nnombre_jours_absence (apres le texte EXACT "Nombre de jour d\'absence", null si absent),\npart_transferable,\npays_destination,\nnom_banque_etranger (première partie après "Compte devise :"),\nnumero_compte_devise_etranger (IBAN après nom_banque_etranger),\nnumero_domiciliation : lire toute la ligne du tableau rectangulaire BNP situé la page document sous "DOMICILIATION IMPORT". Le tableau contient 5 colonnes sur une seule ligne.\n\n--- Si BULLETIN ---\nnom_prenom_salarie,\nmatricule : la première case à coté de "MATRICULE :",\nperiode_bulletin,\nsalaire_base,\nsalaire_brut,\nretenue_ss,\nretenue_irg,\nretenue_mutuelle,\nnet_a_payer\n\n--- Si ANNEXE_I ---\nPrends le temps de lire toute la lettre avant d\'extraire.\nnom_prenom_employe : le nom juste après "Je soussigné" en début de lettre,\ndate_naissance : Né le jj/MM/aaaa,\nrésidence : au moment du recrutement,\ncompte_bancaire_local : titulaire du compte bancaire, e numéro de compte 20 chiffres,\nnom_prenom_signataire : le nom de la personne qui signe en qualité de Responsable des Expatriés,\ndate_signature : la date après "Bethioua le" au format jj/MM/aaaa\n\nRÈGLES :\n- Retourne UNIQUEMENT un JSON valide, sans texte avant ni après\n- Commence toujours par {"type": "..."}\n- Si un champ est absent ou illisible : null\n- Pas de markdown, pas de backticks\n- N\'invente aucun champ supplémentaire\n- Lorsqu\'un texte est écrit entre parenthèses dans le document, extraire exactement ce texte sans modification.\n- Ne pas reformuler, ne pas traduire, ne pas supprimer les parenthèses.\n- Si un champ demande un texte entre parenthèses, retourner uniquement le contenu exact visible.\n- Ignorer les traits verticaux ou "|" des cases\n'

print(f'Prompt : {len(PROMPT_UNIVERSEL)} caractères')


## 8. Traitement d'un PDF (batch GPU)

In [ ]:
def process_pdf(pdf_path: Path, verbose=True) -> dict:
    """
    Traite un PDF avec batch GPU.
    Toutes les pages non-blanches sont envoyées par groupes de GPU_BATCH_SIZE.
    Retourne un dict avec données + stats (tokens, temps).
    """
    import gc
    TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}

    pages    = pdf_to_pages(pdf_path)
    results  = {}
    doublons = []
    pages_par_type = defaultdict(list)
    total_tok_in  = 0
    total_tok_out = 0
    t_dossier     = time.time()

    if verbose:
        print(f'\n📁 {pdf_path.name} — {len(pages)} page(s)')

    # Filtrer pages non-blanches
    pages_actives = [p for p in pages if not is_blank(p['image'])]
    pages_vides   = len(pages) - len(pages_actives)

    if verbose and pages_vides:
        print(f'  {pages_vides} page(s) vide(s) ignorée(s)')

    # Traitement par batch
    for batch_start in range(0, len(pages_actives), GPU_BATCH_SIZE):
        batch  = pages_actives[batch_start:batch_start + GPU_BATCH_SIZE]
        images = [p['image'] for p in batch]

        t_batch = time.time()
        reps    = ask_batch(PROMPT_UNIVERSEL, images)
        elapsed_batch = time.time() - t_batch

        for page, rep in zip(batch, reps):
            i        = page['index']
            data     = parse_json(rep['text'])
            doc_type = data.get('type', 'INCONNU')

            total_tok_in  += rep['tokens_in']
            total_tok_out += rep['tokens_out']

            if doc_type in ('AUTRE', 'INCONNU'):
                if verbose:
                    print(f'  Page {i+1} → {doc_type} — ignorée')
                continue

            data = normalise_doc(doc_type, data)
            pages_par_type[doc_type].append(i + 1)

            if verbose:
                print(f'  Page {i+1} → {doc_type:10s} '
                      f'| tok={rep["tokens_in"]}+{rep["tokens_out"]}')
                for k, v in data.items():
                    if k != 'type' and v is not None:
                        print(f'    {k:25s}: {v}')

            if doc_type in results:
                doublons.append(doc_type)
                if verbose: print(f'    ⚠️  DOUBLON {doc_type}')
                continue

            data['page'] = i + 1
            results[doc_type] = data

        # Purge VRAM entre batches
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pages_trouvees   = sorted(results.keys())
    pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

    return {
        'fichier':          pdf_path.name,
        'version_pipeline': PIPELINE_VERSION,
        'nb_pages':         len(pages),
        'pages_pieces':     ', '.join(f'{t}:' + '+'.join(map(str, sorted(v))) for t, v in sorted(pages_par_type.items(), key=lambda kv: min(kv[1]))) or None,
        'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'temps_total_s':    round(time.time() - t_dossier, 2),
        'tokens_in':        total_tok_in,
        'tokens_out':       total_tok_out,
        'tokens_total':     total_tok_in + total_tok_out,
        'pages_trouvees':   ', '.join(pages_trouvees),
        'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
        'anomalies':        ('DOUBLON: ' + ', '.join(f"{t} (pages {'+'.join(map(str, pages_par_type[t]))})" for t in sorted(set(doublons)))) if doublons else None,
        **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
    }


print('✅ process_pdf OK')

## 9. Export Excel

In [ ]:
SCHEMA = {'META': [('fichier', 'Fichier'),
          ('date_traitement', 'Date traitement'),
          ('temps_total_s', 'Temps (s)'),
          ('tokens_total', 'Tokens total'),
          ('pages_trouvees', 'Pages trouvées'),
          ('pages_pieces', 'Pages par pièce'),
          ('pages_manquantes', 'Pages manquantes'),
          ('anomalies', 'Anomalies')],
 'OV': [('monnaie', 'Monnaie'),
        ('montant_chiffres', 'Montant (chiffres)'),
        ('montant_lettres', 'Montant (lettres)'),
        ('reference_paiement', 'Reference paiement (case 70)'),
        ('periode', 'periode'),
        ('mois', 'mois'),
        ('annee', 'annee'),
        ('tranche', 'Tranche'),
        ('complement_ov', 'complement'),
        ('date_demande', 'Date demande'),
        ('nature_paiement_case', 'Nature paiement (case)'),
        ('compte_donneur_ordre', 'compte Donneur ordre'),
        ('nature_paiement_autre_libelle', 'Devise de transfert'),
        ('beneficiaire_nom', 'Bénéficiaire nom'),
        ('beneficiaire_compte', 'Bénéficiaire compte'),
        ('beneficiaire_adresse', 'Bénéficiaire adresse'),
        ('code_swift_banque_beneficiaire', 'SWIFT'),
        ('nom_banque_beneficiaire', 'Banque bénéficiaire')],
 'ANNEXE_II': [('mois_transfert', 'Mois transfert'),
               ('nom_prenom_travailleur', 'Nom Prénom annex2'),
               ('compte_bancaire_local', 'Compte bancaire'),
               ('salaire_mensuel', 'Salaire mensuel'),
               ('nombre_jours', 'Nombre de jours'),
               ('nombre_jours_absence', 'Nombre jours absence'),
               ('part_transferable', 'Part transférable'),
               ('pays_destination', 'Pays destination'),
               ('nom_banque_etranger', 'Nom_banque'),
               ('numero_compte_devise_etranger', 'Numéro IBAN'),
               ('numero_domiciliation', 'Domiciliation import')],
 'BULLETIN': [('nom_prenom_salarie', 'Nom Prénom Salarie'),
              ('matricule', 'Matricule'),
              ('periode_bulletin', 'Periode bulletin'),
              ('salaire_base', 'Salaire base'),
              ('salaire_brut', 'Salaire brut'),
              ('retenue_ss', 'Retenue SS'),
              ('retenue_irg', 'Retenue IRG'),
              ('retenue_mutuelle', 'Retenue mutuelle'),
              ('net_a_payer', 'Net à payer')],
 'ANNEXE_I': [('nom_prenom_employe', 'Nom prenom client'),
              ('date_naissance', 'Date de naissance'),
              ('résidence', 'Résidence'),
              ('compte_bancaire_local', 'Compte du Client'),
              ('nom_prenom_signataire', 'Nom prenom Signataire'),
              ('date_signature', 'Date de signature')]}

COLORS = {
    'META':      {'header': 'FF1F4E79', 'col': 'FFD6E4F0'},
    'OV':        {'header': 'FF833C00', 'col': 'FFFCE4D6'},
    'ANNEXE_I':  {'header': 'FF375623', 'col': 'FFE2EFDA'},
    'ANNEXE_II': {'header': 'FF203864', 'col': 'FFDAE3F3'},
    'BULLETIN':  {'header': 'FF3F3151', 'col': 'FFEDE7F6'},
}


def create_excel(path: Path, rows: list):
    wb = Workbook()
    ws = wb.active
    ws.title = 'Dossiers'
    all_cols = []
    for groupe, cols in SCHEMA.items():
        for field_key, label in cols:
            all_cols.append((groupe, field_key, label))

    col_idx = 1
    group_map = defaultdict(list)
    for groupe, _, _ in all_cols:
        group_map[groupe].append(col_idx)
        col_idx += 1

    for groupe, cols in group_map.items():
        s, e = cols[0], cols[-1]
        if s < e:
            ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s)
        c.value = groupe
        c.font  = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill  = PatternFill('solid', start_color=COLORS[groupe]['header'])
        c.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 22

    for i, (groupe, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i)
        c.value = label
        c.font  = Font(bold=True, name='Arial', size=9)
        c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = Border(bottom=Side(style='thin'), right=Side(style='hair'))
        ws.column_dimensions[get_column_letter(i)].width = 20
    ws.row_dimensions[2].height = 35
    ws.freeze_panes = ws.cell(row=3, column=len(SCHEMA['META']) + 1)

    for row_num, dossier in enumerate(rows, start=3):
        for col_idx, (groupe, field_key, _) in enumerate(all_cols, start=1):
            val = (dossier.get(field_key) if groupe == 'META'
                   else (dossier.get(groupe) or {}).get(field_key))
            c = ws.cell(row=row_num, column=col_idx)
            c.value = val
            c.font  = Font(name='Arial', size=9)
            c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])
            if isinstance(val, float):
                c.number_format = '0.00'
            if groupe == 'META' and field_key in ('anomalies', 'pages_manquantes') and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')

    wb.save(path)
    print(f'✅ Excel : {path} | {len(rows)} dossiers | {len(all_cols)} colonnes')


print('✅ Export Excel OK')

## 10. Log

In [ ]:
def log(msg: str):
    ligne = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + '\n')


print('✅ Log OK')

## 11. Pipeline complet

In [ ]:
# Dossiers déjà traités = JSON existants
deja_traites = set()
for f in JSON_DIR.glob('*.json'):
    try:
        if json.load(open(f, encoding='utf-8')).get('version_pipeline') == PIPELINE_VERSION:
            deja_traites.add(f.stem)
    except Exception:
        pass   # JSON illisible : le PDF est relu
total_pdfs = len(pdfs)
a_traiter = [p for p in pdfs if p.stem not in deja_traites]
log(f'Total PDFs     : {total_pdfs}')
log(f'Déjà traités   : {len(deja_traites)}')
log(f'À traiter      : {len(a_traiter)}')
log(f'GPU batch size : {GPU_BATCH_SIZE}')

t_total = time.time()
grand_tok_in = 0
grand_tok_out = 0
n_ok = 0
n_err = 0
for num, pdf_path in enumerate(pdfs, start=1):
    if pdf_path.stem in deja_traites:
        log(f'[{num:>4}/{total_pdfs}] ⏩ SKIP : {pdf_path.name}')
        continue
    try:
        result = process_pdf(pdf_path, verbose=False)
        # Sauvegarde JSON immédiate
        json_file = JSON_DIR / f'{pdf_path.stem}.json'
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        grand_tok_in += result.get('tokens_in', 0)
        grand_tok_out += result.get('tokens_out', 0)
        n_ok += 1
        log(f'[{num:>4}/{total_pdfs}] ✅ {pdf_path.name} '
            f'| {result["temps_total_s"]}s '
            f'| tok={result["tokens_total"]} '
            f'| {result["pages_trouvees"]}'
            + (f' | ⚠️ manque: {result["pages_manquantes"]}' if result.get('pages_manquantes') else '')
            + (f' | 🔴 {result["anomalies"]}' if result.get('anomalies') else '')
        )
    except Exception as e:
        n_err += 1
        log(f'[{num:>4}/{total_pdfs}] ❌ {pdf_path.name} — {e}')
        continue

# Reconstruction Excel depuis tous les JSON
log('Génération Excel...')
rows = []
for json_file in sorted(JSON_DIR.glob('*.json')):
    with open(json_file, encoding='utf-8') as f:
        rows.append(json.load(f))

create_excel(EXCEL_PATH, rows)

elapsed = time.time() - t_total
log(f'✅ Terminé en {elapsed:.1f}s ({elapsed/max(1,n_ok):.1f}s/dossier)')
log(f'   Traités      : {n_ok} | Erreurs : {n_err}')
log(f'   Tokens IN    : {grand_tok_in:,}')
log(f'   Tokens OUT   : {grand_tok_out:,}')
log(f'   Tokens TOTAL : {grand_tok_in + grand_tok_out:,}')

## 12. Récupération d'urgence

Si le kernel plante et que `rows` est encore en mémoire, exécuter cette cellule
pour sauvegarder immédiatement en JSON avant de tout perdre.

In [ ]:
# Décommenter et exécuter en cas de crash kernel
# try:
#     print(f'rows en mémoire : {len(rows)}')
#     for result in rows:
#         nom = Path(result['fichier']).stem
#         jf  = JSON_DIR / f'{nom}.json'
#         if not jf.exists():
#             with open(jf, 'w', encoding='utf-8') as f:
#                 json.dump(result, f, ensure_ascii=False, indent=2, default=str)
#     print(f'✅ Sauvegardé')
# except Exception as e:
#     print(f'rows non disponible : {e}')

## 13. Analyse des résultats

In [ ]:
import pandas as pd

if EXCEL_PATH.exists():
    df = pd.read_excel(EXCEL_PATH, header=1)
    print(f'📊 {len(df)} dossiers')

    # Stats tokens
    if 'Tokens total' in df.columns:
        print(f'   Tokens total   : {df["Tokens total"].sum():,.0f}')
        print(f'   Tokens/dossier : {df["Tokens total"].mean():,.0f}')
    if 'Temps (s)' in df.columns:
        print(f'   Temps moyen    : {df["Temps (s)"].mean():.1f}s/dossier')
    print()

    if 'Anomalies' in df.columns:
        anom = df[df['Anomalies'].notna()]
        print(f'🔴 Doublons : {len(anom)}')
        if len(anom): print(anom[['Fichier','Anomalies']].to_string(index=False))
        print()

    if 'Pages manquantes' in df.columns:
        manq = df[df['Pages manquantes'].notna()]
        print(f'⚠️  Incomplets : {len(manq)}')
        if len(manq): print(manq[['Fichier','Pages manquantes']].to_string(index=False))
        print()

    display(df.head(5))
else:
    print('Excel non encore généré.')